In [6]:
import sys, os, json
import pandas as pd
from pathlib import Path

sys.path.insert(1,os.path.abspath('../..'))
from tools.battle import *

log_dir = Path("../../data/replays/gen9-randombattle")
log_dir2 = Path("../../data/replays/gen9-randombattle_2")
log_dir3 = Path("../../data/replays/gen9-randombattle_3")

# helpful to not have to use the full filename
def pull_by_num(num, ns='', parse=False):
    return Battle(f'../../data/replays/gen9-randombattle{ns}/gen9randombattle-{num}.json', parse)

In [13]:
error_ids = [] # battles that have trouble loading

rows=[] # for df
col_names = ['id','w_p1','d_elo','n_turns','time'] # for df

# adv{i}{j} = advantage(team_1_mon_{i}, team_2_mon_{j})
for i in range(6):
    for j in range(6):
        col_names.append(f"adv{i+1}{j+1}")

# easier to call on multiple directories
def foo(file):
    try : 
        bat = Battle(file, parse=True)
        # throw out battles with custom rules and those that last less than 60 seconds
        if not (bat.custom_ruleQ) and not (bat.end_time - bat.start_time < 60): 
            # with file.open() as battle_json:
            #     battle_data = json.load(battle_json)
            
            rows.append([
                bat.id.removeprefix('gen9randombattle-'), 
                int(bat.winner.name == bat.p1.name), # w_p1
                bat.p1.elo0-bat.p2.elo0, # d_elo
                len(bat.STATES), # n_turns
                bat.end_time-bat.start_time # time
                ])
            
            team1 = [FullPokemon(bat.teams_full[0][mon]) for mon in bat.teams_full[0].keys()]
            team2 = [FullPokemon(bat.teams_full[1][mon]) for mon in bat.teams_full[1].keys()]
            
            for i in range(6):
                for j in range(6):
                    rows[-1].append(FullPokemon.advantage(team1[i],team2[j]))
    except : 
        print(f"error with {file.stem}")
        error_ids.append(file.stem)


for file in log_dir.iterdir():
    if file.name.endswith('.json'):
        foo(file)
for file in log_dir2.iterdir():
    if file.name.endswith('.json'):
        foo(file)
for file in log_dir3.iterdir():
    if file.name.endswith('.json'):
        foo(file)

# should have ~12k+ rows, 41 columns
df = pd.DataFrame(rows,columns=col_names)
df.drop_duplicates(subset=['id'], ignore_index=True, inplace=True) # should be redundant, but good to be sure
df 

,id,w_p1,d_elo,n_turns,time,adv11,adv12,adv13,adv14,adv15,...,adv53,adv54,adv55,adv56,adv61,adv62,adv63,adv64,adv65,adv66
0,2631906096,0,-5,33,598,0.715086,1.328708,1.100367,0.649726,-0.168379,...,1.039033,0.872012,0.853608,-0.258471,1.121944,1.654554,1.111303,-0.789109,0.170756,-1.030727
1,2631763570,0,10,23,167,-1.102343,-0.912040,-0.822721,0.397518,-0.967776,...,-0.498879,0.270978,-0.776190,-0.535428,0.476433,-0.809819,0.191290,0.357970,1.173952,1.035264
2,2631369343,1,-69,23,275,-1.108051,0.533011,1.163594,-0.392851,0.407047,...,-0.367578,0.743372,-1.148583,1.080020,1.005441,-1.007017,-0.120664,0.166639,0.305740,0.538460
3,2631529004,1,17,10,123,-0.173456,-0.971856,0.274374,-0.283311,0.510040,...,0.212533,-0.519739,-1.129192,0.670411,1.063704,0.927443,0.942618,-0.156272,1.043459,0.814467
4,2631993792,0,58,24,301,-0.290351,0.339730,0.159254,-0.404944,-0.439897,...,-1.295679,-0.469600,-0.618894,0.692243,1.033497,-0.499937,-0.197355,1.482827,0.203640,1.341519
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12264,2642081603,0,-68,32,436,0.876270,0.287213,-0.330383,0.416327,1.003063,...,0.491708,0.905692,1.115315,-0.612230,-0.361229,1.255143,0.291487,-0.015020,0.941994,-0.537115
12265,2641877736,0,-8,31,239,-0.297292,-0.114010,-0.330454,0.053584,-0.795918,...,-0.256885,-0.306557,-0.257532,0.402548,-0.611551,0.242786,-0.207030,0.274018,-0.406860,0.581827
12266,2642095210,0,-49,23,187,-0.731785,0.812661,-1.467990,-0.786055,1.223906,...,1.019051,0.659662,-0.257854,-0.984054,-0.366654,-0.797029,1.445350,0.551774,-0.422699,0.301263
12267,2642125538,0,4,29,186,1.048859,-0.770079,0.550265,-0.726676,0.234887,...,-0.673489,-0.273518,0.994666,0.248306,0.967479,-0.903486,-0.573505,-0.203632,0.619339,0.263966


# EDA

(Things you can run for some basic plots)

In [ ]:
import ydata_profiling as ydata
profile = ydata.ProfileReport(df, explorative=True)

In [ ]:
import seaborn as sns
sns.pairplot(df, x_vars=['d_elo', 'n_turns', 'time'], y_vars=['w_p1'])

# Modeling

In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import( 
    RandomForestClassifier, 
    ExtraTreesClassifier,
    AdaBoostClassifier
)
from sklearn.model_selection import(
    GridSearchCV, 
    train_test_split, 
    cross_val_score, 
    cross_validate
)

# grab first few columns
X0 = df[['d_elo','n_turns','time']].copy() # include this .copy() or face the ire of pandas
y0 = df.w_p1

# for a given Mon {i}, compute its 'total advantage vs team 2' 
# (could average, but this is taken care of by StandardScaler later)
for i in range(6):
    X0[f'A{i+1}'] = df[df.columns[5+6*i : 5+6*(i+1)]].sum(axis=1)
X0.head()

### "SideQuest:"
I wanted to be extra sure that below when I use `scalar.fit_transform(X)` that each column gets transformed properly

In [29]:
col = 'd_elo'
scaler = StandardScaler()
scaler.fit(X0[[col]])
print(f"mean: {scaler.mean_}, std: {scaler.scale_}")
print()
print(f"transformed {[col]}:\n", scaler.transform(X0[[col]]))

mean: [-13.03472166], std: [57.61981829]

transformed ['d_elo']:
 [[ 0.13944372]
 [ 0.39977081]
 [-0.97128523]
 ...
 [-0.62418243]
 [ 0.29563998]
 [-0.41592076]]


In [30]:
col = 'n_turns'
scaler = StandardScaler()
scaler.fit(X0[[col]])
print(f"mean: {scaler.mean_}, std: {scaler.scale_}")
print()
print(f"transformed {[col]}:\n", scaler.transform(X0[[col]]))

mean: [25.56394164], std: [11.31957512]

transformed ['n_turns']:
 [[ 0.65692027]
 [-0.22650511]
 [-0.22650511]
 ...
 [-0.22650511]
 [ 0.30355012]
 [ 0.74526281]]


In [32]:
scaler = StandardScaler()
X = pd.DataFrame(
    scaler.fit_transform(X0),
    columns=X0.columns,
    index=X0.index
)
X.head()

,d_elo,n_turns,time,A1,A2,A3,A4,A5,A6
0,0.139444,0.656920,1.730388,1.351059,0.651922,-0.717640,-0.429107,0.321864,1.009825
1,0.399771,-0.226505,-0.889161,-2.036219,0.894669,0.368514,0.543812,-0.146766,1.094410
2,-0.971285,-0.226505,-0.232754,0.724133,-0.013846,0.741689,0.828237,0.274772,0.397054
3,0.521257,-1.374958,-1.156586,0.209621,0.506967,0.617313,-0.458304,0.081017,2.097598
4,1.232818,-0.138163,-0.074730,-0.652715,0.810464,0.107595,0.337650,-0.418742,1.520635


In [ ]:
# Looks good! 
# /SideQuest

### Back to work

In [37]:
scaler = StandardScaler()

X = pd.DataFrame(
    scaler.fit_transform(X0),
    columns=X0.columns,
    index=X0.index
)

# From what I understand about SVC, it prefers to have classes {-1,1} vs {0,1}
y = y0.replace(0, -1)

In [38]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    stratify = y, 
    random_state = 502
)

### SVC

In [ ]:
# Trying out some SVC params (took -several- minutes to run, even with all cpu cores)
param_grid = [
    {'kernel': ['linear'], 'C': [0.1, 1, 10, 100]},
    {'kernel': ['rbf'], 'C': [0.1, 1, 10, 100], 'gamma': [0.01, 0.1, 1, 'scale']},
    {'kernel': ['poly'], 'C': [0.1, 1, 10], 'degree': [2, 3], 'gamma': [0.1, 1, 'scale']}
]

grid = GridSearchCV(
    SVC(), 
    param_grid, 
    cv=5, 
    n_jobs=-1 #NOTE:`n_jobs=-1` will use all available cpu cores; melt cpu at own risk
)
grid.fit(X_train, y_train)

print(grid.best_params_, grid.best_score_)

### RandomForest

In [ ]:
param_grid = [{
    'n_estimators': [50,100,200], 
    'max_depth': [2,3,5], 
    'criterion': ["gini","entropy","log_loss"]
}]

gridRF = GridSearchCV(
    RandomForestClassifier(), 
    param_grid, 
    cv=5, 
    n_jobs=-1 #NOTE:`n_jobs=-1` will use all available cpu cores; melt cpu at own risk
)
gridRF.fit(X_train, y_train)

print(gridRF.best_params_, gridRF.best_score_)

### AdaBoost

In [ ]:
param_gridAB = [{
    'estimator': ['LogisticRegression','DecisionTreeClassifier'], 
    'n_estimators': [25, 50, 100]
    'learning_rate': [0.5, 1, 2]
}]

gridAB = GridSearchCV(
    AdaboostClassifier(), 
    param_grid, 
    cv=5, 
    n_jobs=-1 #NOTE:`n_jobs=-1` will use all available cpu cores; melt cpu at own risk
)
gridAB.fit(X_train, y_train)

print(gridAB.best_params_, gridAB.best_score_)

### (Pulled some stuff from `sklearn` docs)

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

from sklearn.datasets import make_hastie_10_2
from sklearn.metrics import accuracy_score, make_scorer
from sklearn.model_selection import GridSearchCV
scoring = {"AUC": "roc_auc", "accuracy": "accuracy", "f1": "f1"}

In [ ]:
model = SVC(
    C=100,
    kernel='linear'
)
cvt = cross_validate(model, X_train, y_train, cv=5, n_jobs=-1, scoring=scoring)

In [ ]:
print(cvt['test_f1'].mean())